![image.png](https://i.imgur.com/a3uAqnb.png)

In [4]:
import torch
import torch.nn as nn

# This model is designed for very small images
# Challenges > Small resolution >> we want a simple CNN model

# Definition of my model.
class MyModel(nn.Module):
  def __init__(self, num_classes):
    super(MyModel, self).__init__()

    self.features_extractor = nn.Sequential(
     nn.Conv2d(3,32, kernel_size = 3, padding=1), # 32 by 32 by 32 [h, w, out_channels]
     nn.ReLU(),
     nn.MaxPool2d(2), # 16 by 16 by 32
     nn.Conv2d(32, 64, kernel_size = 3, padding=1), # 16 by 16 by 64
     nn.ReLU(),
     nn.MaxPool2d(2) # 8 by 8 by 64
    )

    self.nn_classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(8 * 8 * 64, 128),
        nn.ReLU(),
        nn.Linear(128, num_classes)
    )

  def forward(self, x):
    image_features = self.features_extractor(x)
    yhat = self.nn_classifier(image_features)
    return yhat

x = torch.randn(1, 3, 32, 32)
model = MyModel(10)
yhat = model(x)
print(yhat.shape)

torch.Size([1, 64, 8, 8])
torch.Size([1, 10])


# 1. Small dataset, similar distribution → Freeze extractor, train classifier

# •Keep the feature extraction part fixed and fine-tune the classifier part of the network


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
num_classes = 5
model = models.resnet18(pretrained=True)
for param in model.parameters():
    param.requires_grad = False
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, num_classes)
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

print("\n\nScenario 1: Feature extractor frozen.")
print("Trainable params:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)



Scenario 1: Feature extractor frozen.
Trainable params:
fc.weight
fc.bias


# 2. Large dataset, similar distribution → Fine-tune entire network
# Fine tune both the feature extractor and the classifier part of the network

In [ ]:
model = models.resnet18(pretrained=True)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, num_classes)
for param in model.parameters():
    param.requires_grad = True
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
print("\n\nScenario 2: All layers will be fine-tuned.")
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)



Scenario 2: All layers will be fine-tuned.
Trainable params: 11179077


# 3. Small dataset, different distribution → Frozen extractor + SVM

In [ ]:
from sklearn.svm import SVC
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

model = models.resnet18(pretrained=True)
for param in model.parameters():
    param.requires_grad = False
model.fc = torch.nn.Identity()  # Remove classifier for feature extraction
print("\n\nScenario 3: Using frozen feature extractor + external SVM.")
print("No trainable params in CNN backbone.")
# def extract_features(dataloader, model, device="cpu"):
#     features, labels = [], []
#     model.to(device).eval()
#     with torch.no_grad():
#         for x, y in dataloader:
#             x = x.to(device)
#             feat = model(x).cpu()
#             features.append(feat)
#             labels.append(y)
#     return torch.cat(features).numpy(), torch.cat(labels).numpy()
# transform = transforms.Compose([
#     transforms.Resize((224,224)),
#     transforms.ToTensor()
# ])
# dataset = ImageFolder("data/train", transform=transform)
# dataloader = DataLoader(dataset, batch_size=32, shuffle=False)
# X, y = extract_features(dataloader, model)
# svm = SVC(kernel="linear")
# svm.fit(X, y)



Scenario 3: Using frozen feature extractor + external SVM.
No trainable params in CNN backbone.


# 4. Large dataset, different distribution → Fine-tune whole model

In [ ]:
model = models.resnet18(pretrained=True)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, num_classes)

for param in model.parameters():
    param.requires_grad = True
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
print("\n\nScenario 4: Whole network (feature extractor + classifier) will be fine-tuned.")
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))



Scenario 4: Whole network (feature extractor + classifier) will be fine-tuned.
Trainable params: 11179077
